# Pruebas de hipótesis de las medidas de los componentes ERP

## Funciones

In [54]:
import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.api as sm
from scipy.stats import chi2_contingency
from statsmodels.formula.api import ols
from statsmodels.sandbox.stats.multicomp import multipletests


In [55]:
# Función principal para realizar ANCOVA completo con cálculo de EMMs, SEs, ICs
#  y tamaños del efecto

def test_ancova_full(data, dv, between):

    # =======================================================================
    # 1. PREPARACIÓN DE DATOS Y AJUSTE DEL MODELO FINAL
    # =======================================================================

    # Cargar el dataframe y limpiar los datos
    df = data.dropna(subset=[dv, between])

    # Fórmula del ANCOVA final: C(type) + age + scholarship + C(gender)
    ancova_model_formula = f'{dv} ~ C({between}) + age + scholarship + C(gender)'
    ancova_model = ols(ancova_model_formula, data=df).fit()
    # Generar la tabla ANOVA Tipo II (para obtener las Sumas de Cuadrados)
    ancova_results = sm.stats.anova_lm(ancova_model, typ=2)

    # Imprimir la tabla ANOVA (para referencia)
    print("--- Tabla ANCOVA del Modelo Final ---")
    print(ancova_results)
    print("------------------------------------")

    # =======================================================================
    # 2. CÁLCULO DE PARÁMETROS Y MATRIZ DE CONTRASTES (L)
    # =======================================================================

    # Parámetros para fijar las covariables:
    mean_age = df['age'].mean()
    mean_scholarship = df['scholarship'].mean()

    # Proporción de 'M' (que es la categoría dummy que Statsmodels crea)
    gender_counts = df['gender'].value_counts()
    prop_M = gender_counts['M'] / gender_counts.sum()

    # Orden de los coeficientes del modelo
    coef_names = ancova_model.params.index.tolist()
    # ['Intercept', C(type)[T.nonvictim], C(type)[T.victim], age, scholarship, C(gender)[T.M]]  # noqa: E501

    # Inicializar matriz de contrastes (L)
    L = np.zeros((3, len(coef_names)))

    # Valores comunes a todos los contrastes (fijando covariables)
    common_values = {
        'Intercept': 1,
        'age': mean_age,
        'scholarship': mean_scholarship,
        'C(gender)[T.M]': prop_M
    }

    # =======================================================================
    # 3. Construir la matriz L: 1 fila por cada nivel de between
    # =======================================================================

    # EMM para Grupo de Referencia
    cat_vars = df[between].unique().tolist()

    for name, val in common_values.items():
        L[0, coef_names.index(name)] = val

    #cat_vars = cat_vars[1:]  # Eliminar el grupo de referencia

    # EMM para level_1
    for name, val in common_values.items():
        L[1, coef_names.index(name)] = val
    L[1, coef_names.index(f'C({between})[T.{cat_vars[1]}]')] = 1

    # EMM para level_2
    for name, val in common_values.items():
        L[2, coef_names.index(name)] = val
    L[2, coef_names.index(f'C({between})[T.{cat_vars[2]}]')] = 1

    # =======================================================================
    # 4. CÁLCULO DE EMMs, SEs, e ICs
    # =======================================================================

    # Usar t_test para obtener el estimate (EMM), SE (sd), e Intervalo de Confianza (conf_int)  # noqa: E501
    t_tests_emms = ancova_model.t_test(L)

    # 5. Formatear resultados
    comparisons = df[between].unique().tolist()
    ci_lower = t_tests_emms.conf_int().T[0]
    ci_upper = t_tests_emms.conf_int().T[1]

    medias_ajustadas = pd.DataFrame({
        'type': comparisons,
        'Media_Ajustada (M_adj)': t_tests_emms.effect.round(4),
        'Error_Estandar (EE)': t_tests_emms.sd.round(4),
        'IC_95%_Inferior': ci_lower.round(4),
        'IC_95%_Superior': ci_upper.round(4)
    })

    print("--- Medias Ajustadas, Error Estándar e Intervalos de Confianza (95%) ---")
    print(medias_ajustadas)
    print("------------------------------------")

    # =======================================================================
    # 5. CÁLCULO DE TAMAÑO DE EFECTO: PARTIAL ETA-SQUARED (η²p)
    # =======================================================================

    # 1. Extraer la Suma de Cuadrados del Error (SS_Residual)
    ss_residual = ancova_results.loc['Residual', 'sum_sq']

    # 2. Calcular la Eta-Squared Parcial (η²p) para cada efecto
    # Fórmula: η²p = SS_Efecto / (SS_Efecto + SS_Residual)

    eta_p_results = {}
    for effect in [f'C({between})', 'C(gender)', 'age', 'scholarship']:
        ss_effect = ancova_results.loc[effect, 'sum_sq']
        eta_p = ss_effect / (ss_effect + ss_residual)
        eta_p_results[effect] = eta_p

    # 3. Consolidar resultados
    eta_squared_df = pd.DataFrame.from_dict(
        eta_p_results, orient='index', columns=['Eta_Squared_Parcial (η²p)']
    ).reset_index().rename(columns={'index': 'Efecto'})

    print("--- Tamaños del Efecto: Partial Eta-Squared (η²p) ---")
    print(eta_squared_df.round(4))

    return ancova_model

In [56]:
# Función para verificar los supuestos del ANCOVA mediante un modelo con interacciones

def verify_ancova_assumptions(data, dv, between):
    # Cargar el dataframe y eliminar la fila con el valor faltante en IAT_score
    df = data.dropna(subset=[dv, between])

    # Definición y ajuste del modelo de interacción
    ancova_interaction_model = ols( f'{dv} ~ C({between}) * age + C({between}) * scholarship + C({between}) * C(gender)', 
    data=df).fit()

    # --- VERIFICACIÓN DEL SUPUESTO (TABLA ANOVA) ---
    # Usamos ANOVA Tipo II para evaluar la significancia de las interacciones
    # (C(type):age), (C(type):scholarship), y (C(type):C(gender)).
    ancova_interaction_results = sm.stats.anova_lm(ancova_interaction_model, typ=2)

    print("--- ANCOVA con interacciones (Tabla de verificación de supuestos) ---")
    print("El supuesto se cumple si el p-valor (PR(>F)) de las interacciones es > 0.05")
    print(ancova_interaction_results)
    return

In [57]:
# Función para realizar pruebas post-hoc con corrección de Bonferroni en medias ajustadas  # noqa: E501

def posthoc_contrasts_bonferroni(data, dv, between, ancova_model):

    # =======================================================================
    # 2. DEFINICIÓN Y APLICACIÓN DE CONTRASTES POST-HOC
    # =======================================================================

    # Identificar el orden de los coeficientes del modelo para construir la matriz L.
    # El orden es: [Intercept, C(type)[T.nonvictim], C(type)[T.victim], age, scholarship]  # noqa: E501
    coef_names = ancova_model.params.index.tolist()

    # Crear la matriz de contrastes (L) para las 3 comparaciones por pares
    # La matriz L tendrá 3 filas (comparaciones) y 5 columnas (coeficientes)
    L = np.zeros((3, len(coef_names)))

    cat_vars = data[between].unique().tolist()
    cat_vars = cat_vars[1:]  # Eliminar el grupo de referencia

    # Índice de los coeficientes de interés
    idx_level_1 = coef_names.index(f'C({between})[T.{cat_vars[0]}]')
    idx_level_2 = coef_names.index(f'C({between})[T.{cat_vars[1]}]')
    L[0, idx_level_1] = 1 
    L[1, idx_level_2] = 1
    L[2, idx_level_1] = 1
    L[2, idx_level_2] = -1

    # =======================================================================
    # 3. PRUEBA T Y CORRECCIÓN POR COMPARACIONES MÚLTIPLES
    # =======================================================================

    # Realizar las pruebas t con la matriz de contrastes
    t_tests = ancova_model.t_test(L)

    # Extraer los p-valores sin ajustar
    raw_p_values = t_tests.pvalue

    # Aplicar la corrección de Bonferroni
    reject, p_adjusted, _, _ = multipletests(
        raw_p_values,
        alpha=0.05,
        method='bonferroni'
    )

    # =======================================================================
    # 4. CREACIÓN Y VISUALIZACIÓN DE RESULTADOS
    # =======================================================================

    cat_vars = data[between].unique().tolist()

    comparisons = [
        f'{cat_vars[1]} vs. {cat_vars[0]}',
        f'{cat_vars[2]} vs. {cat_vars[0]}',
        f'{cat_vars[2]} vs. {cat_vars[1]}'
    ]

    results_df = pd.DataFrame({
        'Comparación': comparisons,
        'Diferencia_Media_Ajustada': t_tests.effect.round(4),
        'Estadístico_t': t_tests.tvalue.round(4),
        'p_valor_sin_ajustar': raw_p_values.round(4),
        'p_valor_Bonferroni': p_adjusted.round(4),
        'Significativo (alpha=0.05)': reject
    })

    print("--- Pruebas Post-Hoc (Bonferroni) en Medias Ajustadas ---")
    print(results_df)
    return

## Análisis covariables

In [58]:
data_subjects_erps = pd.read_csv('data_subjects.csv')
data_subjects_erps = data_subjects_erps.rename(columns={
    'Unnamed: 0': 'subject'
})

data_subjects_erps.head()

,subject,type,group,victim_self,exposure_level,age,gender,scholarship,laterality,IAT_score,IAT_level,mean_co,mean_in,mean_dif
0,21100,excombatant,exguerrilla,yes,high,19,F,11,D,0.319044,High,0.586812,0.537628,0.049184
1,21101,excombatant,exparamilitar,yes,high,46,M,11,D,-0.070836,Medium,0.463325,1.170137,-0.706813
2,21102,excombatant,exparamilitar,yes,low,31,M,11,D,0.488057,High,-4.757545,-3.940181,-0.817364
3,21105,excombatant,exparamilitar,yes,high,48,M,8,D,0.258352,High,0.410431,0.404159,0.006272
4,21107,excombatant,exguerrilla,no,high,25,M,11,D,-0.119566,Low,0.256688,0.392556,-0.135867


### Age

In [59]:
# Normalidad de age por type
pg.normality(
    data=data_subjects_erps, 
    dv='age', 
    group='type', 
    method='shapiro', 
    alpha=0.05
    )

,W,pval,normal
type,,,
excombatant,0.975969,0.397055,True
nonvictim,0.958400,0.664618,True
victim,0.960358,0.496554,True


In [60]:
# Homocedasticity de age por type
pg.homoscedasticity(
    data=data_subjects_erps, 
    dv='age', 
    group='type', 
    method='levene', 
    alpha=0.05
    )

,W,pval,equal_var
levene,3.071996,0.051569,True


In [61]:
# ANOVA de age por type
pg.anova(
    data=data_subjects_erps, 
    dv='age', 
    between='type', 
    effsize='np2'
    )

,Source,ddof1,ddof2,F,p_unc,np2
0,type,2,84,2.672338,0.074963,0.059821


**Conclusión**: No hay diferencias significativas en la edad por tipo de actor.

### Scholarship

In [62]:
# Normalidad de scholarship por type
pg.normality(
    data=data_subjects_erps, 
    dv='scholarship', 
    group='type', 
    method='shapiro', 
    alpha=0.05
    )

,W,pval,normal
type,,,
excombatant,0.945034,0.021413,False
nonvictim,0.766030,0.001392,False
victim,0.868774,0.007428,False


In [63]:
# Homocedasticity de scholarship por type
pg.homoscedasticity(
    data=data_subjects_erps, 
    dv='scholarship', 
    group='type', 
    method='levene', 
    alpha=0.05
    )

,W,pval,equal_var
levene,3.751866,0.027498,False


In [64]:
# Kruskal para scholarship por type
pg.kruskal(
    data=data_subjects_erps, 
    dv='scholarship', 
    between='type'
    )

,Source,ddof1,H,p_unc
Kruskal,type,2,10.042729,0.006596


**Conclusión**: Sí hay diferencias significativas en los años de estudio por tipo de actor.

### Sex

In [65]:
# prueba chi-squared type vs gender
contingency_table = pd.crosstab(
    data_subjects_erps['type'], 
    data_subjects_erps['gender']
    )

chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi2: {chi2}, p-value: {p}, dof: {dof}")

Chi2: 31.557011988011993, p-value: 1.4043711983548123e-07, dof: 2


**Conclusión**: Sí hay diferencias significativas en el sexo por tipo de actor.

## Análisis IAT

In [66]:
# Tabla de contingencia type vs IAT_level
pd.crosstab(data_subjects_erps['type'], data_subjects_erps['IAT_level'], margins=True)

IAT_level,High,Low,Medium,All
type,,,,
excombatant,15,19,16,50
nonvictim,3,7,4,14
victim,11,1,10,22
All,29,27,30,86


In [67]:
dv = 'IAT_score'
between = 'type'

ancova_model = test_ancova_full(
    data=data_subjects_erps, 
    dv=dv, 
    between=between
)

--- Tabla ANCOVA del Modelo Final ---
               sum_sq    df         F    PR(>F)
C(type)      0.498590   2.0  2.636368  0.077838
C(gender)    0.007024   1.0  0.074283  0.785902
age          0.017788   1.0  0.188117  0.665656
scholarship  0.457950   1.0  4.842955  0.030643
Residual     7.564800  80.0       NaN       NaN
------------------------------------
--- Medias Ajustadas, Error Estándar e Intervalos de Confianza (95%) ---
          type  Media_Ajustada (M_adj)  Error_Estandar (EE)  IC_95%_Inferior  \
0  excombatant                 -0.0117               0.0477          -0.1067   
1    nonvictim                 -0.0396               0.0869          -0.2125   
2       victim                  0.2117               0.0823           0.0479   

   IC_95%_Superior  
0           0.0832  
1           0.1333  
2           0.3755  
------------------------------------
--- Tamaños del Efecto: Partial Eta-Squared (η²p) ---
        Efecto  Eta_Squared_Parcial (η²p)
0      C(type)            

In [68]:
verify_ancova_assumptions(
    data=data_subjects_erps,
    dv=dv,
    between=between
)

--- ANCOVA con interacciones (Tabla de verificación de supuestos) ---
El supuesto se cumple si el p-valor (PR(>F)) de las interacciones es > 0.05
                       sum_sq    df         F    PR(>F)
C(type)              0.498590   2.0  2.634989  0.078442
C(gender)            0.014494   1.0  0.153197  0.696624
C(type):C(gender)    0.016035   2.0  0.084741  0.918839
age                  0.007238   1.0  0.076500  0.782869
C(type):age          0.404102   2.0  2.135631  0.125397
scholarship          0.461549   1.0  4.878462  0.030290
C(type):scholarship  0.252188   2.0  1.332783  0.269998
Residual             7.001104  74.0       NaN       NaN


**Conclusiones:**: No hay efecto de type cuando se ajusta por covariables.

## Análisis componente ERP

In [69]:
dv = 'mean_dif'
between = 'type'

ancova_model = test_ancova_full(
    data=data_subjects_erps, 
    dv=dv, 
    between=between
)

--- Tabla ANCOVA del Modelo Final ---
                 sum_sq    df         F    PR(>F)
C(type)       14.297530   2.0  4.983867  0.009093
C(gender)      5.058306   1.0  3.526473  0.063995
age            1.997301   1.0  1.392448  0.241446
scholarship    0.004662   1.0  0.003250  0.954678
Residual     116.184865  81.0       NaN       NaN
------------------------------------
--- Medias Ajustadas, Error Estándar e Intervalos de Confianza (95%) ---
          type  Media_Ajustada (M_adj)  Error_Estandar (EE)  IC_95%_Inferior  \
0  excombatant                  0.1193               0.1851          -0.2489   
1    nonvictim                  0.8197               0.3283           0.1664   
2       victim                 -0.7037               0.3213          -1.3430   

   IC_95%_Superior  
0           0.4876  
1           1.4730  
2          -0.0644  
------------------------------------
--- Tamaños del Efecto: Partial Eta-Squared (η²p) ---
        Efecto  Eta_Squared_Parcial (η²p)
0      C(type)

In [70]:
verify_ancova_assumptions(
    data=data_subjects_erps,
    dv=dv,
    between=between
)

--- ANCOVA con interacciones (Tabla de verificación de supuestos) ---
El supuesto se cumple si el p-valor (PR(>F)) de las interacciones es > 0.05
                         sum_sq    df         F    PR(>F)
C(type)               14.297530   2.0  4.919726  0.009826
C(gender)              2.907721   1.0  2.001071  0.161326
C(type):C(gender)      1.434993   2.0  0.493776  0.612288
age                    1.878758   1.0  1.292947  0.259125
C(type):age            5.034477   2.0  1.732344  0.183871
scholarship            0.015699   1.0  0.010804  0.917494
C(type):scholarship    1.460837   2.0  0.502668  0.606938
Residual             108.981150  75.0       NaN       NaN


**Conclusión:** si hay diferencias por tipo de actor.

In [71]:
posthoc_contrasts_bonferroni(
    data=data_subjects_erps,
    dv=dv,
    between=between,
    ancova_model=ancova_model
)

--- Pruebas Post-Hoc (Bonferroni) en Medias Ajustadas ---
                 Comparación  Diferencia_Media_Ajustada  Estadístico_t  \
0  nonvictim vs. excombatant                     0.7004         1.8737   
1     victim vs. excombatant                    -0.8230        -2.0299   
2       victim vs. nonvictim                     1.5235         3.1565   

   p_valor_sin_ajustar  p_valor_Bonferroni  Significativo (alpha=0.05)  
0               0.0646              0.1937                       False  
1               0.0456              0.1369                       False  
2               0.0022              0.0067                        True  


**Conclusiones**: Hay diferencias significativas entre no-víctimas y víctimas, ya ajustando por las covariables. En no-víctimas las amplitudes promedio de los ensayos del bloque congruente tienden a ser más grandes (positivas) que las del incongruente. En víctimas sucede lo contrario.

## ANOVA Mixto

In [72]:
def anova_mixto(data, within, between):

    data_long = pd.melt(
        data, 
        id_vars=['subject', between], 
        value_vars=within, 
        var_name='condition', 
        value_name='amplitude'
        )


    anova = pg.mixed_anova(
        dv='amplitude', 
        within='condition', 
        between=between, 
        subject='subject', 
        data=data_long,
        correction='auto'
        )
    
    posthoc = pg.pairwise_tests(
        dv='amplitude', 
        between=between, 
        within='condition', 
        data=data_long, 
        parametric=True, 
        subject='subject', 
        padjust='holm',
        effsize='cohen'
        )

    return anova, posthoc

In [73]:
# ANOVA MIXTO

within = ['mean_co', 'mean_in']
between = 'type'

omnibus, posthoc = anova_mixto(data=data_subjects_erps, within=within, between=between)
print(omnibus)

        Source        SS  DF1  DF2        MS         F     p_unc       np2  \
0         type  0.273336    2   84  0.136668  0.080698  0.922544  0.001918   
1    condition  0.044415    1   84  0.044415  0.059238  0.808297  0.000705   
2  Interaction  6.816419    2   84  3.408209  4.545664  0.013352  0.097660   

   eps  
0  NaN  
1  1.0  
2  NaN  


In [74]:
print(posthoc)

           Contrast condition            A          B Paired Parametric  \
0         condition         -      mean_co    mean_in   True       True   
1              type         -  excombatant  nonvictim  False       True   
2              type         -  excombatant     victim  False       True   
3              type         -    nonvictim     victim  False       True   
4  condition * type   mean_co  excombatant  nonvictim  False       True   
5  condition * type   mean_co  excombatant     victim  False       True   
6  condition * type   mean_co    nonvictim     victim  False       True   
7  condition * type   mean_in  excombatant  nonvictim  False       True   
8  condition * type   mean_in  excombatant     victim  False       True   
9  condition * type   mean_in    nonvictim     victim  False       True   

          T        dof alternative     p_unc    p_corr p_adjust   BF10  \
0  0.233936  86.000000   two-sided  0.815591       NaN      NaN  0.122   
1 -0.296686  25.989773   t